In [ ]:
#same as double_loci_restrict3, but trying to read npz files.

import os
import glob
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.stats import rankdata
from scipy.ndimage import gaussian_filter
from scipy import ndimage
import scipy.io as sio

#from starhopping import basic
import sys
from starhoppingpy import basic
from starhoppingpy import functions
from ipywidgets import interact, FloatSlider
from starhoppingpy import zfuncs as zf
#%matplotlib widget
#%matplotlib notebook

name='hd141569'


In [ ]:
#Rescale ref PSF to match Science PSF.

import os
DATA_DIR = "/path/to/SORT/dayYYYY-MM-DD-cidNNNNNNN-IRDIS"  # set to your sorted data folder
os.chdir(DATA_DIR)

%load_ext autoreload
%autoreload 2
from starhoppingpy.sh_funcs import sph_rescale_ref_psf

sph_rescale_ref_psf("imcbasic1.save.npz", "imcbasic2.save.npz", "imcbasic2b.save.npz")


In [ ]:
#Read basic calibrated image FITS cubes.
os.chdir(DATA_DIR)

sdat = np.load("imcbasic1.save.npz", allow_pickle=True)
rdat = np.load("imcbasic2b.save.npz", allow_pickle=True)
cubeS = sdat["imc"].copy() 
cubeR = rdat["imc"].copy()
img0=cubeS[0,:,:]
angs = sdat["pas"] # reads rotation angles list
angr = rdat["pas"] # reads rotation angles list
print('total sky rotation (deg) =', zf.anglediff(angs[-1],angs[0]))


In [ ]:
# define regions and
# remove background slopes from cubes.

ginx=basic.get_mask

#w0=ginx(img0,6,10)
w0=ginx(img0,0,4)
w1=ginx(img0,70,110) # 
w2=ginx(img0,90,120)  # 
w3=ginx(img0,7,120)
wb=ginx(img0,120,240) # background region
wsb=ginx(img0,6,120) # all signal and some background

wm1=w0 | w1
wm2=w0 | w2

nums = np.shape(cubeS)[0] # number of science images
numr = np.shape(cubeR)[0] # number of ref images

cubeS = functions.remove_bgnd_slope(cubeS)
cubeR = functions.remove_bgnd_slope(cubeR)



In [ ]:
# SKIP FOR DISKS or EXTENDED SOURCES
# spatial filtering all images to keep point-like sources only.

cubeS[:] = np.array([zf.getfreq(cubeS[i], 4) for i in range(cubeS.shape[0])])
cubeR[:] = np.array([zf.getfreq(cubeR[i], 4) for i in range(cubeR.shape[0])])
    

In [ ]:
#compare the median of science and ref cubes
smed = np.median(cubeS, axis=0)
rmed = np.median(cubeR, axis=0)

dmed=functions.zscalediff2(smed, rmed, w0,w1)
zf.zdisp3(smed,rmed,dmed,limfrac=0.1,zoom=1)
#factor to balance rms in inner and outer regions for LOCI match.
rfac = functions.rgn_fac(smed,w0,w1)
print('Need zscalediff 2, \nas ratio of optim scalefac for rgn w0 and w1  = ', rfac)
print('No of ref images:',cubeR.shape[0],'\nNo of sci images:', cubeS.shape[0])

In [ ]:
cubeSn = cubeS.copy()
cubeRn = cubeR.copy()

diffs = cubeSn.copy()
diffr = cubeRn.copy()

snrs = np.zeros(nums)


for i in range(nums):
    img = cubeSn[i,:,:]
    ref=functions.loci_operation(cubeSn[i,:,:], cubeRn, wsb)
    diff = cubeSn[i,:,:] - ref
    zf.zdisp3(cubeS[i, :, :],ref,diff, limfrac=0.1,zoom=2)
    diffs[i,:,:]= ndimage.rotate(diff,angs[i],reshape=False)
    #diffr[i,:,:]= ndimage.rotate(ref-rmed ,angs[i],reshape=False)

for i in range(numr):
    diff = cubeRn[i,:,:] - rmed
    diffr[i,:,:]= ndimage.rotate(diff,angr[i],reshape=False)


In [ ]:
#First reduc and mask making.
    
redsci = np.median(diffs,axis=0)
refres = np.median(diffr,axis=0)
rmsmap = zf.rmsmap_ann(refres)

#wback = np.where(redsci < rmsmap)
#redsci[wback] = 0

dd = zf.dist_circle(redsci.shape[0])
radius_mask = (dd <= 110) & (dd >= 12)

lev = 1
lev2 = 40

bmask = (redsci < lev*rmsmap) & radius_mask
smask = (redsci > lev2*rmsmap) & radius_mask

bmask = bmask | w1
smask = smask & ~w1

csmask = gaussian_filter(smask.astype(float), sigma=20)
bmask[csmask > 0.5] = False

zf.zdisp3(redsci, rmsmap, np.maximum(redsci,rmsmap), limfrac=0.1,zoom=2,samescale=True)
zf.zdisp2(smask.astype(int),bmask.astype(int))


In [ ]:
# make an average bgnd mask

mask = bmask*0.0

for i in range(nums):
    mask = mask + ndimage.rotate(bmask.astype(float),-angs[i],reshape=False)

mask = mask/nums
mask = mask >= 0.5

nmask = mask & wsb

zf.zdisp(nmask.astype(int))

In [ ]:
# This time use the masks and redo the reduc
# Remove the REF-based LOCI-estimate of the median SCI PSF from both SCI and REF cubes
# match the detside when doing this.

#the new mask is 'nmask'

cubeSn = cubeS.copy()
cubeRn = cubeR.copy()


#mkeys = ['lambdas', 'stokes', 'polfilts', 'detside']
#mkeys = ['polfilts', 'detside']
mkeys = ['detside']

inx_arr = zf.keygroups(sdat, mkeys)
print(inx_arr)

key_arr = zf.firstkeyvals(sdat, mkeys ,inx_arr)
print(key_arr)



#define some empty output dicts
res0, res1, res2 =  ({},) * 3 

numref=16
i=0
for key_values in key_arr:
    print("Processing key values:", key_values)
    
    wc = zf.wherekeys(rdat, key_values)
    rimcc = cubeRn[wc, :, :]
    
    wbest = inx_arr[i]
    #wbest = np.intersect1d(inx_arr[i],sorted_indices[0:30])
    simcc = cubeSn[wbest, :, :]  

    # Compute the median along axis=0 of the subset
    smed = np.median(simcc, axis=0)
    rmed = np.median(rimcc, axis=0)

    # Perform operations using smed, rimc_subset, wm, numref

    nrefs = np.min([rimcc.shape[0], numref]) # use numrefs or max available.
    oc = functions.good_refs2(smed, rimcc, nmask, nrefs)
    ref = functions.loci2(smed, rimcc[oc[0],:,:], w0, nmask)
    
    zf.zdisp3(smed,ref,smed-ref, limfrac=0.1,zoom=2)
    #zf.zdisp(smed, limfrac=0.1,zoom=2)
    #input('...')
    
    junk=functions.zscalediff2(smed,ref,w0,nmask,res0)
    
    #zf.zdisp3(smed,res0['mimg'],smed-res0['mimg'], limfrac=0.1,zoom=2)
        
    print(wbest)
    for idx in wbest:
        cubeSn[idx]=functions.zscalediff2(cubeS[idx, :, :],res0['mimg'],w0,w1, res1)
        #zf.zdisp3(cubeS[idx, :, :],res1['mimg'],cubeSn[idx, :, :], limfrac=0.1,zoom=2)
    for idx in wc:
        cubeRn[idx]=functions.zscalediff2(cubeR[idx, :, :],res0['mimg'],w0,w1, res2)
        #zf.zdisp3(cubeR[idx, :, :],res2['mimg'],cubeRn[idx, :, :], limfrac=0.1,zoom=2)
    i=i+1

In [ ]:
#ranking the reference images for each science images (saved in rms array)
#but using the background mask this time, which will be rotated to the sci ori

nrefs = np.min([cubeRn.shape[0]-1, 16])
oc2 = functions.good_refs_mask(cubeSn, cubeRn, wsb, nrefs, mask=bmask, pas=angs)    


In [ ]:
# making the second science diff cubes

diffs = cubeSn.copy()
snrs = np.zeros(nums)

for i in range(nums):
    bmaskr = ndimage.rotate(bmask,-angs[i],reshape=False) > 0.5
    smaskr = ndimage.rotate(smask,-angs[i],reshape=False) > 0.5

    img = cubeSn[i,:,:]
    ref=functions.loci_operation(cubeSn[i,:,:], cubeRn[oc2[:,i],:,:], bmaskr)
    diff = cubeSn[i,:,:] -ref
    snrs[i] = np.sum(diff[np.where(smaskr)])/np.std(diff[np.where(bmaskr)])
    #zf.zdisp3(cubeSn[i, :, :],ref,diff, limfrac=0.1,zoom=2)

    diff2 = ndimage.rotate(diff,angs[i],reshape=False)
    diffs[i,:,:]= diff2

redsci = np.median(diffs[np.where(snrs > np.max(snrs)*0.7)],axis=0)
    

In [ ]:
#Stack the reduced difference frames
np.median(snrs),np.min(snrs), np.max(snrs)

redsci = np.median(diffs[np.where(snrs > np.max(snrs)*0.65)],axis=0)
redsci[ginx(img0,0,8)]=0
zf.writefits(name+'_red1.fits',redsci)

zf.zdisp(redsci, limfrac=0.1,zoom=1.5)


In [ ]:
# Make another round of masks for signal and bgnd regions
rmed2 = np.median(cubeRn,axis=0)

for i in range(numr):
    diffr[i,:,:] = ndimage.rotate(cubeRn[i,:,:]-rmed2,angr[i],reshape=False)

refres2 = np.median(diffr,axis=0)
rmsmap2 = zf.rmsmap_ann(refres2)
zf.zdisp2(redsci,rmsmap2,zoom=2)

radius_mask = (dd <= 120) & (dd >= 12)

lev = 1
lev2 = 5

bmask = (redsci < lev*rmsmap) & radius_mask
smask = (redsci > lev2*rmsmap) & radius_mask

#bmask = bmask
#smask = smask & ~w1
zf.zdisp3(redsci, rmsmap, np.maximum(redsci,2*rmsmap), limfrac=0.1,zoom=2,samescale=True)

zf.zdisp2(smask.astype(int),bmask.astype(int))